# ROGII - GRU-refiner v12 permutation importance (real 3 test wells)

Which of the 2 new v12 features (pf_ancc_std, pf_z_std) actually drives the win over v7 (5.001 vs 5.509 pooled on the real 3 test wells)? Zeros each feature in turn using the already-trained checkpoints -- no training needed.

Run: Input = competition dataset + rogii-gru-v12-assets + rogii-gru-refiner-v12 (both private). CPU is fine. Internet off. Run All.


In [ ]:
"""Permutation importance for GRU-refiner v12's 2 new features (pf_ancc_std, pf_z_std), on the REAL 3
test wells specifically (the decisive metric all session) -- which of the two PF posterior-spread
features is actually driving v12's win over v7 (5.001 vs 5.509 pooled)? Zeroing a feature (rather than
shuffling) is used since these are the only 3 wells available here and shuffling within n=3 is
meaningless; zeroing approximates "feature absent" the same way v8's permutation-importance script
tested its disagreement features. No training needed -- reuses the already-trained v12 checkpoints.
"""
import sys, os, glob, pickle, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

_c = glob.glob('/kaggle/input/**/*__horizontal_well.csv', recursive=True)
_t = [p for p in _c if 'train' in p.lower()]
_c = _t if _t else _c
TRAIN_DIR = os.path.dirname(_c[0]) if _c else 'd:/ROGII/data/train'
_assets = glob.glob('/kaggle/input/**/proxy_slim.pkl', recursive=True)
ASSET_DIR = os.path.dirname(_assets[0]) if _assets else '.'
_ckpt = glob.glob('/kaggle/input/**/gru_refiner_v12_fold0_seed0.pt', recursive=True)
CKPT_DIR = os.path.dirname(_ckpt[0]) if _ckpt else '.'
sys.path.insert(0, ASSET_DIR)
import pf_ancc_source as pfs
print(f'TRAIN_DIR={TRAIN_DIR}  ASSET_DIR={ASSET_DIR}  CKPT_DIR={CKPT_DIR}', flush=True)

SEQ_LEN = 500
CFG = dict(d=96, layers=2, n_folds=5, seed=11)
N_FEAT = 26
REAL3 = ['000d7d20', '00bbac68', '00e12e8b']


def interp_nan(a):
    a = a.copy(); n = len(a); idx = np.arange(n); m = np.isnan(a)
    if m.all(): return np.zeros(n)
    a[m] = np.interp(idx[m], idx[~m], a[~m]); return a


def build_well_raw(wid):
    hw = pd.read_csv(f'{TRAIN_DIR}/{wid}__horizontal_well.csv')
    tw = pd.read_csv(f'{TRAIN_DIR}/{wid}__typewell.csv').sort_values('TVT')
    tw_tvt = tw['TVT'].values.astype(float); tw_gr = tw['GR'].fillna(tw['GR'].mean()).values.astype(float)
    km = hw['TVT_input'].notna()
    return hw, tw_tvt, tw_gr, km


def feats_for_cut(hw, tw_tvt, tw_gr, km, pf_ancc_std, pf_z_std):
    km_idx = np.where(km.values)[0]
    last_i = km_idx[-1]
    last_tvt = float(hw['TVT_input'].iloc[last_i]); last_Z = float(hw['Z'].iloc[last_i]); last_MD = float(hw['MD'].iloc[last_i])
    gr = interp_nan(hw['GR'].values.astype(float))
    gap = hw['GR'].isna().values.astype(np.float32)
    X_ = hw['X'].values.astype(float); Y = hw['Y'].values.astype(float)
    Z = hw['Z'].values.astype(float); MD = hw['MD'].values.astype(float)
    mdd = np.gradient(MD); mdd[mdd == 0] = 1
    kg = gr[km_idx]; ktvt = hw['TVT_input'].values[km_idx]
    twk = np.interp(ktvt, tw_tvt, tw_gr); v = np.isfinite(kg) & np.isfinite(twk)
    a, b = (np.polyfit(kg[v], twk[v], 1) if v.sum() >= 20 else (1., 0.))
    cal = gr * a + b
    cs = pd.Series(cal)
    sm5 = cs.rolling(5, center=True, min_periods=1).mean().values
    sm15 = cs.rolling(15, center=True, min_periods=1).mean().values
    sm41 = cs.rolling(41, center=True, min_periods=1).mean().values
    dog1 = sm5 - sm15; dog2 = sm15 - sm41
    grad = np.gradient(gr)
    rstd = pd.Series(gr).rolling(21, center=True, min_periods=1).std().fillna(0).values
    dzdmd = np.gradient(Z) / mdd
    head = np.arctan2(np.gradient(Y), np.gradient(X_) + 1e-9)
    fwd_mean = cs.rolling(40, min_periods=1).mean().shift(-40).bfill().ffill().values
    fwd_std = cs.rolling(40, min_periods=1).std().shift(-40).bfill().ffill().fillna(0).values
    F_ = {}
    F_['md_since'] = MD - last_MD
    F_['gr'] = gr; F_['cal_gr'] = cal
    F_['sm5'] = sm5; F_['sm15'] = sm15; F_['sm41'] = sm41
    F_['dog1'] = dog1; F_['dog2'] = dog2
    F_['grad'] = grad; F_['rstd'] = rstd
    F_['gap'] = gap
    F_['z'] = Z - last_Z; F_['dzdmd'] = dzdmd
    F_['fwd_mean'] = fwd_mean; F_['fwd_std'] = fwd_std
    for o in (-20, -10, -5, 0, 5, 10, 20):
        F_['tda%d' % o] = gr - np.interp(last_tvt + o, tw_tvt, tw_gr)
    F_['sin_azi'] = np.sin(head); F_['cos_azi'] = np.cos(head)
    ORDER = ['md_since', 'gr', 'cal_gr', 'sm5', 'sm15', 'sm41', 'dog1', 'dog2', 'grad', 'rstd', 'gap',
              'z', 'dzdmd', 'fwd_mean', 'fwd_std',
              'tda-20', 'tda-10', 'tda-5', 'tda0', 'tda5', 'tda10', 'tda20', 'sin_azi', 'cos_azi']
    X = np.stack([F_[c] for c in ORDER], axis=1)
    n_rows = X.shape[0]
    ev_mask_full = hw['TVT_input'].isna().values
    std_a = np.zeros(n_rows, dtype=np.float32); std_z = np.zeros(n_rows, dtype=np.float32)
    std_a[ev_mask_full] = pf_ancc_std; std_z[ev_mask_full] = pf_z_std
    X = np.concatenate([X, std_a[:, None], std_z[:, None]], axis=1)
    return X


class GRURefinerV12(nn.Module):
    def __init__(self, n_in, d, layers):
        super().__init__()
        self.inp = nn.Linear(n_in, d)
        self.gru = nn.GRU(d, d, num_layers=layers, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(0.0), nn.Linear(d, 1))
    def forward(self, X):
        h = F.gelu(self.inp(X))
        h, _ = self.gru(h)
        return self.head(h)[..., 0]


if __name__ == '__main__':
    slim_path = glob.glob('/kaggle/input/**/proxy_slim.pkl', recursive=True)
    slim_path = slim_path[0] if slim_path else 'proxy_slim.pkl'
    proxy = pickle.load(open(slim_path, 'rb'))

    # replicate the EXACT fold split used in v12 training (sorted all-773 wids, seed=11 shuffle, 5-way split)
    parent_wids = sorted(proxy.keys())
    rng = np.random.default_rng(CFG['seed'])
    order = np.array(parent_wids); rng.shuffle(order)
    folds = np.array_split(order, CFG['n_folds'])
    fold_of = {}
    for fi, f in enumerate(folds):
        for w in f: fold_of[w] = fi
    print('fold assignment for real3:', {w: fold_of.get(w) for w in REAL3}, flush=True)

    results = {}
    ns = {}
    for wid in REAL3:
        fi = fold_of[wid]
        hw, tw_tvt, tw_gr, km = build_well_raw(wid)
        ev_mask = hw['TVT_input'].isna().values
        sp45 = proxy[wid]['sp45'].astype(np.float64)
        true = hw['TVT'].values.astype(float)[ev_mask]
        n_ev = int(ev_mask.sum())
        ns[wid] = n_ev
        assert len(sp45) == n_ev, (wid, len(sp45), n_ev)

        _, pf_ancc_std = pfs.run_pf_ancc(hw, tw_tvt, tw_gr)
        _, pf_z_std = pfs.run_pf_z(hw, tw_tvt, tw_gr)

        X = feats_for_cut(hw, tw_tvt, tw_gr, km, pf_ancc_std, pf_z_std)
        Xev = X[ev_mask]
        n = Xev.shape[0]
        src = np.arange(n); dst = np.linspace(0, n - 1, SEQ_LEN)
        Xr = np.stack([np.interp(dst, src, Xev[:, c]) for c in range(N_FEAT)], axis=1).astype(np.float32)

        norm = pickle.load(open(f'{CKPT_DIR}/gru_v12_norm_fold{fi}.pkl', 'rb'))
        mean, std = norm['mean'], norm['std']

        def predict(Xr_local):
            Xn = (Xr_local - mean) / std
            preds = []
            for seed_i in range(2):
                net = GRURefinerV12(N_FEAT, CFG['d'], CFG['layers'])
                net.load_state_dict(torch.load(f'{CKPT_DIR}/gru_refiner_v12_fold{fi}_seed{seed_i}.pt', map_location='cpu'))
                net.eval()
                with torch.no_grad():
                    Xt = torch.tensor(Xn[None], dtype=torch.float32)
                    r = net(Xt)[0].numpy()
                src_dst = np.linspace(0, n - 1, SEQ_LEN)
                pred_full = np.interp(np.arange(n), src_dst, r)
                preds.append(sp45 + pred_full)
            return np.mean(preds, axis=0)

        base_pred = predict(Xr)
        Xr_noancc = Xr.copy(); Xr_noancc[:, 24] = 0.0
        Xr_nopz = Xr.copy(); Xr_nopz[:, 25] = 0.0
        Xr_neither = Xr.copy(); Xr_neither[:, 24] = 0.0; Xr_neither[:, 25] = 0.0

        rmse = lambda p: float(np.sqrt(np.mean((p - true) ** 2)))
        r_sp45 = rmse(sp45); r_base = rmse(base_pred); r_noancc = rmse(predict(Xr_noancc))
        r_nopz = rmse(predict(Xr_nopz)); r_neither = rmse(predict(Xr_neither))
        results[wid] = dict(fold=fi, sp45=r_sp45, base=r_base, zero_ancc=r_noancc, zero_pz=r_nopz, zero_both=r_neither)
        print(f'{wid} (fold{fi}) n={n_ev}: sp45={r_sp45:.3f} v12_full={r_base:.3f} '
              f'zero_pf_ancc_std={r_noancc:.3f} zero_pf_z_std={r_nopz:.3f} zero_both={r_neither:.3f}', flush=True)

    print(flush=True)
    for key in ['sp45', 'base', 'zero_ancc', 'zero_pz', 'zero_both']:
        num = sum(results[w][key] ** 2 * ns[w] for w in REAL3)
        den = sum(ns[w] for w in REAL3)
        print(f'POOLED {key}: {(num / den) ** 0.5:.4f}', flush=True)

